In [2]:
!pip install -q imbalanced-learn

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = pd.read_csv(
    "/content/drive/MyDrive/Internship project/05_processed_text_dataset.csv"
)

print(df.shape)
df.head()

(1206, 11)


,Message_ID,Person,Chat_Name,Timestamp,Sender,Message,Risk_Label,Risk_Category,Confidence,Processed_Message,Tokens
0,1,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:15:00,Vishnu,"Da, report kandille?",Normal,NaN,High,da report kandille,"['da', 'report', 'kandille']"
1,2,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:17:00,You,Kandu. Ellam okay alle?,Normal,NaN,High,kandu ellam okay alle,"['kandu', 'ellam', 'okay', 'alle']"
2,3,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:18:00,Vishnu,Mostly okay. Pakshe aa last item kurachu stran...,Suspicious,Coded Language,Medium,mostly okay pakshe aa last item kurachu strang...,"['mostly', 'okay', 'pakshe', 'aa', 'last', 'it..."
3,4,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:20:00,You,Entha issue?,Normal,NaN,High,entha issue,"['entha', 'issue']"
4,5,ARAVIND MENON,Chat with Vishnu,2026-01-05 06:21:00,Vishnu,Numbers match cheyyunnilla. Randu places il di...,Suspicious,Coded Language,Medium,number match cheyyunnilla randu place il diffe...,"['number', 'match', 'cheyyunnilla', 'randu', '..."


In [5]:
X = df["Processed_Message"].fillna("")
y = df["Risk_Label"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [8]:
print("Before SMOTE")
print(y_train.value_counts())

Before SMOTE
Risk_Label
Normal        723
Suspicious    185
High Risk      56
Name: count, dtype: int64


In [9]:
smote = SMOTE(
    random_state=42
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_tfidf,
    y_train
)

In [10]:
print("After SMOTE")
print(pd.Series(y_train_smote).value_counts())

After SMOTE
Risk_Label
Normal        723
Suspicious    723
High Risk     723
Name: count, dtype: int64


In [11]:
smote_model = LinearSVC(
    class_weight="balanced",
    random_state=42
)

smote_model.fit(
    X_train_smote,
    y_train_smote
)

LinearSVC(class_weight='balanced', random_state=42)

In [12]:
smote_pred = smote_model.predict(
    X_test_tfidf
)

In [13]:
print("Accuracy:",
      accuracy_score(y_test, smote_pred))

print()

print(classification_report(
    y_test,
    smote_pred
))

Accuracy: 0.6033057851239669

              precision    recall  f1-score   support

   High Risk       0.24      0.36      0.29        14
      Normal       0.82      0.68      0.74       182
  Suspicious       0.25      0.37      0.30        46

    accuracy                           0.60       242
   macro avg       0.43      0.47      0.44       242
weighted avg       0.67      0.60      0.63       242



In [14]:
cm = confusion_matrix(
    y_test,
    smote_pred
)

pd.DataFrame(
    cm,
    index=smote_model.classes_,
    columns=smote_model.classes_
)

,High Risk,Normal,Suspicious
High Risk,5,4,5
Normal,11,124,47
Suspicious,5,24,17


In [16]:
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Linear SVM",
        "Linear SVM + SMOTE"
    ],
    "Accuracy": [
        0.70,
        0.74,
        round(accuracy_score(y_test, smote_pred), 2)
    ],
    "Macro_F1": [
        0.49,
        0.52,
        0.44
    ]
})

comparison

,Model,Accuracy,Macro_F1
0,Logistic Regression,0.70,0.49
1,Linear SVM,0.74,0.52
2,Linear SVM + SMOTE,0.60,0.44


In [ ]:
#An additional experiment using SMOTE was conducted to address class imbalance. Although the training dataset became perfectly balanced,
#the resulting Linear SVM model showed lower accuracy and lower F1-scores across all classes.
#Therefore, the non-SMOTE Linear SVM model was selected as the final mode